In [ ]:
# Set global font properties for all matplotlib plots
import matplotlib.pyplot as pl

# Configure global font settings
pl.rcParams['font.size'] = 17
pl.rcParams['axes.titlesize'] = 19
pl.rcParams['axes.labelsize'] = 17
pl.rcParams['xtick.labelsize'] = 16
pl.rcParams['ytick.labelsize'] = 16
pl.rcParams['legend.fontsize'] = 15
pl.rcParams['figure.titlesize'] = 21


# Optional: Change font family (uncomment if you want to change the font type)
# pl.rcParams['font.family'] = 'serif'  # Options: 'serif', 'sans-serif', 'monospace'
# pl.rcParams['font.serif'] = ['Times New Roman']  # Specific serif font
# pl.rcParams['font.sans-serif'] = ['Arial']  # Specific sans-serif font

print("Global font settings updated for all plots")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

# Project root (run notebook from CHEOPSPR folder, or set this to the project path)
PROJECT_ROOT = os.getcwd()
os.chdir(PROJECT_ROOT)

# Planet to load: "HD202772Ab", "KELT7b", or "TOI1518b"
PLANET = "TOI1518b"

# Aperture to prefer (25 or 24); will use first match if exact not found
APERTURE = 25

COLORS = [
    "#773baf",  # rich purple
    "#74af3b",  # fresh green
    "#e51284",  # vivid pink
    "#B56576",  # dusty rose
    "#008080",  # deep teal
    "#e8d119",  # golden yellow
    "#C76B6B",  # muted red
    "#624D5B",  # aubergine grey
]

In [ ]:
def find_lightcurve_files(planet_name, aperture=None):
    """Find all *Lightcurve-R*_V0300.fits under planet folder (recursive)."""
    planet_dir = os.path.join(PROJECT_ROOT, planet_name)
    if not os.path.isdir(planet_dir):
        return []
    found = []
    for root, _, files in os.walk(planet_dir):
        for f in files:
            if "Lightcurve-R" in f and f.endswith("_V0300.fits"):
                path = os.path.join(root, f)
                # parse aperture from filename (e.g. Lightcurve-R25_V0300.fits)
                try:
                    r = int(f.split("Lightcurve-R")[1].split("_")[0])
                except Exception:
                    r = None
                found.append((path, r, root))
    # Prefer requested aperture, then sort by (aperture, path) for stable order
    if aperture is not None:
        preferred = [x for x in found if x[1] == aperture]
        if preferred:
            found = preferred
    found.sort(key=lambda x: (x[1] or 0, x[0]))
    return found


def load_one_lightcurve(filepath):
    """Load time, flux, flux_err and systematics from a single FITS lightcurve."""
    with fits.open(filepath) as hdul:
        data = hdul[1].data
        print(data.columns.names)  # Print available columns for debugging
        time = np.asarray(data["BJD_TIME"], dtype=float)
        flux = np.asarray(data["FLUX"], dtype=float)
        flux_err = np.asarray(data["FLUXERR"], dtype=float)
        roll_angle = np.asarray(data["ROLL_ANGLE"], dtype=float)
        centroid_x = np.asarray(data["CENTROID_X"], dtype=float)
        centroid_y = np.asarray(data["CENTROID_Y"], dtype=float)
        smearing_lc = np.asarray(data["SMEARING_LC"], dtype=float)
        background = np.asarray(data["BACKGROUND"], dtype=float)     
    med = np.nanmedian(flux)
    if med <= 0:
        med = 1.0
    flux_n = flux / med
    flux_err_n = flux_err / med
    return {
        "time": time,
        "flux": flux_n,
        "flux_err": flux_err_n,
        "roll_angle": roll_angle,
        "centroid_x": centroid_x,
        "centroid_y": centroid_y,
        "smearing_lc": smearing_lc,
        "background": background,
    }


def load_all_visits(planet_name, aperture=None):
    """Load all lightcurves for a planet; group by directory (one visit per dir)."""
    file_list = find_lightcurve_files(planet_name, aperture)
    if not file_list:
        return []
    # Group by directory so we get one "visit" per folder
    by_dir = {}
    for path, ap, root in file_list:
        if root not in by_dir:
            by_dir[root] = []
        by_dir[root].append((path, ap))
    visits = []
    for root in sorted(by_dir.keys()):
        # Use first file in folder (or prefer requested aperture)
        items = by_dir[root]
        if aperture is not None:
            best = [x for x in items if x[1] == aperture]
            if best:
                items = best
        path = items[0][0]
        visit_name = os.path.basename(root)
        d = load_one_lightcurve(path)
        visits.append({"name": visit_name, **d})
    return visits

In [ ]:
# Load all visits for the selected planet
visits = load_all_visits(PLANET, APERTURE)
print(f"Planet: {PLANET}")
print(f"Number of visits: {len(visits)}")
for v in visits:
    print(f"  - {v['name']}: {len(v['time'])} points")

In [ ]:
if not visits:
    print(f"No lightcurve FITS found under {PLANET}. Check path and that files are extracted.")
else:
    n_visits = len(visits)
    colors = [COLORS[i % len(COLORS)] for i in range(n_visits)]

    fig, ax = plt.subplots(1, 1, figsize=(12, 5))
    for i, v in enumerate(visits):
        ax.errorbar(
            v["time"],
            v["flux"],
            yerr=v["flux_err"],
            fmt=".",
            ms=4,
            alpha=0.85,
            color=colors[i],
            label=v["name"],
            capsize=1.5,
        )
    ax.set_xlabel("BJD (days)")
    ax.set_ylabel("Normalized flux")
    ax.set_title(f"{PLANET} — Light curves")
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)
    ax.axhline(1.0, color="gray", ls="--", alpha=0.5)
    #ax.set_ylim(0.995, 1.005)
    fig.tight_layout()
    plt.show()

In [ ]:
# Now perform sigma clipping on flux and background, and re-plot using clipped data
if not visits:
    print(f"No lightcurve FITS found under {PLANET}. Check path and that files are extracted.")
else:
    # Check if flux has already been clipped to avoid re-clipping
    if not all('flux_clipped' in v for v in visits):
        # sigma-clipping parameters
        sigma = 5.0
        max_iters = 5

        # perform sigma clipping per visit and replace arrays with clipped versions
        clip_keys = [
            "time",
            "flux",
            "flux_err",
            "roll_angle",
            "centroid_x",
            "centroid_y",
            "smearing_lc",
            "background",
        ]
        for v in visits:
            f = np.asarray(v["flux"], dtype=float)
            good = np.isfinite(f) & np.isfinite(v["time"]) & np.isfinite(v["flux_err"])
            if not np.any(good):
                v["mask"] = good
                continue
            mask = good.copy()
            for _ in range(max_iters):
                if not np.any(mask):
                    break
                med = np.nanmedian(f[mask])
                std = np.nanstd(f[mask])
                new_mask = good & (np.abs(f - med) <= sigma * std)
                if new_mask.sum() == mask.sum():
                    mask = new_mask
                    break
                mask = new_mask
            # apply clipping to arrays
            for k in clip_keys:
                if k in v:
                    v[k] = np.asarray(v[k])[mask]
            # reset mask to all-True for the new (clipped) arrays
            v["mask"] = np.ones_like(v["flux"], dtype=bool)
        
        # Mark flux as clipped
        for v in visits:
            v['flux_clipped'] = True

    # Check if background has already been clipped to avoid re-clipping
    if not all('bg_clipped' in v for v in visits):
        # Sigma clip data points with high background values (above 5 sigma)
        sigma_bg = 5.0
        max_iters_bg = 5

        for v in visits:
            bg = np.asarray(v["background"], dtype=float)
            good = np.isfinite(bg)
            if not np.any(good):
                v["mask_bg"] = good
                continue
            mask = good.copy()
            for _ in range(max_iters_bg):
                if not np.any(mask):
                    break
                med = np.nanmedian(bg[mask])
                std = np.nanstd(bg[mask])
                new_mask = good & (bg <= med + sigma_bg * std)
                if new_mask.sum() == mask.sum():
                    mask = new_mask
                    break
                mask = new_mask
            # Apply clipping to all arrays (replace with clipped versions)
            for k in clip_keys:
                if k in v:
                    v[k] = np.asarray(v[k])[mask]
            # Reset mask to all-True for the new (clipped) arrays
            v["mask"] = np.ones_like(v["flux"], dtype=bool)
        
        # Mark background as clipped
        for v in visits:
            v['bg_clipped'] = True

    # plotting using clipped data
    n_visits = len(visits)
    colors = [COLORS[i % len(COLORS)] for i in range(n_visits)]

    fig, ax = plt.subplots(1, 1, figsize=(12, 5))
    for i, v in enumerate(visits):
        m = v.get("mask", np.ones_like(v["flux"], dtype=bool))
        ax.errorbar(
            v["time"][m],
            v["flux"][m],
            yerr=v["flux_err"][m],
            fmt=".",
            ms=4,
            alpha=0.85,
            color=colors[i],
            label=v["name"],
            capsize=1.5,
        )
    ax.set_xlabel("BJD (days)")
    ax.set_ylabel("Normalized flux")
    ax.set_title(f"{PLANET} — Light curves (sigma-clipped)")
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)
    ax.axhline(1.0, color="gray", ls="--", alpha=0.5)
    #ax.set_ylim(0.995, 1.005)
    fig.tight_layout()
    plt.show()


### One subplot per visit: light curve + systematics (stacked, semi-transparent)

In [ ]:
if visits:
    from matplotlib.transforms import blended_transform_factory
    n = len(visits)
    ncol = min(1, n)
    nrow = (n + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(6 * ncol, 6 * nrow), squeeze=False)
    axes = axes.flatten()
    def _norm(y):
        y = np.nan_to_num(y, nan=np.nanmedian(y))
        r = np.nanmax(y) - np.nanmin(y)
        if r <= 0:
            return np.zeros_like(y)
        return (y - np.nanmin(y)) / r
    sys_alpha = 0.5
    band = 0.10
    offsets = [0.8, 0.6, 0.4, 0.2, 0.0]  # vertical offsets for systematics
    sys_names = ["Roll angle", "Centroid X", "Centroid Y", "Smearing LC", "Background"]
    for i, v in enumerate(visits):
        ax = axes[i]
        c = COLORS[i % len(COLORS)]
        t, f, fe = v["time"], v["flux"], v["flux_err"]
        ax.errorbar(t, f, yerr=fe, fmt=".", ms=3, alpha=0.9, color=c, capsize=1, label="flux")
        roll = np.asarray(v["roll_angle"])
        cx = np.asarray(v["centroid_x"])
        cy = np.asarray(v["centroid_y"])
        sm = np.asarray(v["smearing_lc"])
        bc = np.asarray(v["background"])
        y_roll = offsets[0] + _norm(roll) * band
        y_cx = offsets[1] + _norm(cx) * band
        y_cy = offsets[2] + _norm(cy) * band
        y_sm = offsets[3] + _norm(sm) * band
        y_bc = offsets[4] + _norm(bc) * band
        ax.plot(t, y_roll, ".", ms=2, alpha=sys_alpha, color=c, label="roll")
        ax.plot(t, y_cx, ".", ms=2, alpha=sys_alpha, color=c, label="centroid_x")
        ax.plot(t, y_cy, ".", ms=2, alpha=sys_alpha, color=c, label="centroid_y")
        ax.plot(t, y_sm, ".", ms=2, alpha=sys_alpha, color=c, label="smearing")
        ax.plot(t, y_bc, ".", ms=2, alpha=sys_alpha, color=c, label="background")
        ax.set_title(v["name"])
        ax.set_xlabel("BJD (days)")
        ax.set_ylabel("Normalized flux")
        #ax.set_ylim(-0.02, 1.08)
        trans = blended_transform_factory(ax.transAxes, ax.transData)
        for j, name in enumerate(sys_names):
            yc = offsets[j] + band / 2
            ax.text(-0.14, yc, name, transform=trans, va="center", ha="right")
        ax.grid(True, alpha=0.3)
        ax.axhline(1.0, color="gray", ls="--", alpha=0.5)
    for j in range(len(visits), len(axes)):
        axes[j].set_visible(False)
    fig.suptitle(f"{PLANET} — Light curves + systematics by visit")
    fig.tight_layout()
    plt.show()

In [ ]:
# Additional plots: flux vs each systematics for all visits

if visits:
    sys_keys = [
        ("roll_angle", "Roll angle"),
        ("centroid_x", "Centroid X"),
        ("centroid_y", "Centroid Y"),
        ("smearing_lc", "Smearing LC"),
        ("background", "Background"),
    ]
    n_sys = len(sys_keys)
    n_vis = len(visits)
    fig, axes = plt.subplots(n_sys, n_vis, figsize=(5 * n_vis, 3.5 * n_sys), squeeze=False)
    for j, (sys_key, sys_label) in enumerate(sys_keys):
        for i, v in enumerate(visits):
            ax = axes[j, i]
            diag = np.asarray(v[sys_key])
            flux = np.asarray(v["flux"])
            ax.plot(diag, flux, ".", ms=3, alpha=0.7, color=COLORS[i % len(COLORS)])
            ax.set_xlabel(sys_label)
            if i == 0:
                ax.set_ylabel("Normalized flux")
            else:
                ax.set_ylabel("")
            if j == 0:
                ax.set_title(v["name"])
            ax.grid(True, alpha=0.3)
    fig.suptitle(f"{PLANET} — Flux vs systematics for all visits")
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


### One subplot per visit: light curves only

In [ ]:
if visits:
    n = len(visits)
    ncol = min(1, n)
    nrow = (n + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(6 * ncol, 5 * nrow), squeeze=False)
    axes = axes.flatten()
    for i, v in enumerate(visits):
        ax = axes[i]
        c = COLORS[i % len(COLORS)]
        print(np.median(np.diff(v["time"])*24*60))
        ax.errorbar(v["time"], v["flux"], yerr=v["flux_err"], fmt=".", ms=3, alpha=0.9, color=c, capsize=1)
        ax.set_title(v["name"])
        ax.set_xlabel("BJD (days)")
        ax.set_ylabel("Normalized flux")
        #ax.set_ylim(0.995, 1.005)
        ax.grid(True, alpha=0.3)
        ax.axhline(1.0, color="gray", ls="--", alpha=0.5)
    for j in range(len(visits), len(axes)):
        axes[j].set_visible(False)
    fig.suptitle(f"{PLANET} — Light curves by visit")
    fig.tight_layout()
    plt.show()

### Secondary eclipse fit for all visits simultaneously (joint)

In [ ]:
import numpy as np
import dynesty
from dynesty import utils as dyfunc
import batman
import os
import pickle
import corner
import matplotlib.pyplot as plt

result_base = f"dynesty_results/TOI1518b/dynesty_result_joint_all_visits.pkl"
os.makedirs("dynesty_results/TOI1518b", exist_ok=True)

# normalize systematics (zero-mean, unit std)
def norm(x):
    x = np.asarray(x, dtype=float)
    xm = np.nanmedian(x)
    xs = np.nanstd(x)
    if xs <= 0:
        return x - xm
    return (x - xm) / xs

# Known orbital parameters (from exoplanet archive))
t0_known = 2459983.791942     # Time of inferior conjunction (BJD)
P_known = 1.90261131          # days
rp_known = 0.09939            # Rp/R*
a_known = 4.109               # a/R*
inc_known = 77.626            # degrees

# batman setup helper: build light curve for given eclipse parameters (t0, depth)
def batman_eclipse_model(tarr, t_secondary, fp, t0=t0_known, per=P_known, rp=rp_known, a=a_known, inc=inc_known):    
    # period, a, inclination are fixed (obtained from exoplanet archive: most precise values)
    # we only fit t_secondary and fp here
    params = batman.TransitParams()        # object to store transit parameters
    params.t_secondary = t_secondary       # eclipse time
    params.fp = fp                         # planet-to-star flux ratio (eclipse depth)
    params.t0 = t0                         # Time of inferior conjunction (BJD)
    params.per = per                       # Orbital period (days)
    params.rp = rp                         # Planet-to-star radius ratio
    params.a = a                           # Semi-major axis (in units of stellar radii)
    params.inc = inc                       # Orbital inclination (degrees)
    params.ecc = 0.0                       # Eccentricity (0 for circular)
    params.w = 90.0                        # Argument of periastron (degrees)
    params.u = [0.0, 0.0]                  # Limb darkening coefficients
    params.limb_dark = "quadratic"
    m = batman.TransitModel(params, tarr, transittype="secondary")
    return m.light_curve(params)

# priors setup for common parameters
min_value = -0.01 # -10
max_value = 0.01 # 10

t_span = t.max() - t.min()
t_secondary_min, t_secondary_max = t.min() - 0.1 * t_span, t.max() + 0.1 * t_span
fp_min, fp_max = 0.0, 0.1  # allow up to 10% eclipse depth
jitter_min, jitter_max = 0.5, 3.0

c_offset_min, c_offset_max = min_value, max_value
slope_min, slope_max = min_value, max_value
c_cy_min, c_cy_max = min_value, max_value
c_cx_min, c_cx_max = min_value, max_value
c_roll_sin2_min, c_roll_sin2_max = min_value, max_value
c_roll_sin3_min, c_roll_sin3_max = min_value, max_value
c_roll_cos3_min, c_roll_cos3_max = min_value, max_value

# --- precompute normalized systematics once ---
visits_norm = []
for visit in visits:
    v = dict(visit)  # shallow copy
    roll_rad = np.radians(visit["roll_angle"])
    v["roll_sin"] = np.sin(roll_rad)
    v["roll_sin2"] = v["roll_sin"] ** 2
    v["roll_sin3"] = np.sin(3.0 * roll_rad)
    v["roll_cos3"] = np.cos(3.0 * roll_rad)
    v["t"] = np.asarray(visit["time"])
    v["cx_norm"] = norm(np.asarray(visit["centroid_x"]))
    v["cy_norm"] = norm(np.asarray(visit["centroid_y"]))
    v["sm_norm"] = norm(np.asarray(visit["smearing_lc"]))
    v["bc_norm"] = norm(np.asarray(visit["background"]))
    visits_norm.append(v)

# Joint fitting routine for all visits
def model_joint_flux(theta, visits):
    """
    Fit all visits simultaneously with shared t_secondary and fp, but visit-specific systematics.
    visits: list of dicts with keys 'time', 'flux', 'flux_err', 'roll_angle', 'centroid_x', 'centroid_y', 'smearing_lc', 'background'
    theta: [t_secondary, fp, offset_v1, slope_v1, jitter_v1, c_cy_v1, c_cx_v1, c_roll_sin2_v1, offset_v2, slope_v2, jitter_v2, c_roll_sin3_v2, c_roll_cos3_v2]
    """

    t_secondary, fp, offset_v1, slope_v1, _jitter_v1, c_cy_v1, c_cx_v1, c_roll_sin2_v1, offset_v2, slope_v2, _jitter_v2, c_roll_sin3_v2, c_roll_cos3_v2 = theta

    model_names = []
    model_flux_list = []
    bat_list = []
    corr_list = []

    for i, visit in enumerate(visits):
        bat = batman_eclipse_model(visit['time'], t_secondary, fp)

        if i == 0:  # Visit 1
            corr = offset_v1 + slope_v1 * (visit["t"] - np.nanmedian(visit["t"])) + c_cy_v1 * visit["cy_norm"] + c_cx_v1 * visit["cx_norm"] + c_roll_sin2_v1 * visit["roll_sin2"]
            m = bat + corr
            model_names.append("base__cy__cx__roll_sin2")
        else:  # Visit 2
            corr = offset_v2 + slope_v2 * (visit["t"] - np.nanmedian(visit["t"])) + c_roll_sin3_v2 * visit["roll_sin3"] + c_roll_cos3_v2 * visit["roll_cos3"]
            m = bat + corr
            model_names.append("base__roll_sin3__roll_cos3")

        model_flux_list.append(m)
        bat_list.append(bat)
        corr_list.append(corr)

    return model_names, model_flux_list, bat_list, corr_list


def prior_joint(unit):
    u0, u1, u2, u3, u4, u5, u6, u7, u8, u9, u10, u11, u12 = unit

    t_secondary = t_secondary_min + u0 * (t_secondary_max - t_secondary_min)
    fp = fp_min + u1 * (fp_max - fp_min)

    offset_v1 = c_offset_min + u2 * (c_offset_max - c_offset_min)
    slope_v1 = slope_min + u3 * (slope_max - slope_min)
    jitter_v1 = jitter_min + u4 * (jitter_max - jitter_min)
    c_cy_v1 = c_cy_min + u5 * (c_cy_max - c_cy_min)
    c_cx_v1 = c_cx_min + u6 * (c_cx_max - c_cx_min)
    c_roll_sin2_v1 = c_roll_sin2_min + u7 * (c_roll_sin2_max - c_roll_sin2_min)

    offset_v2 = c_offset_min + u8 * (c_offset_max - c_offset_min)
    slope_v2 = slope_min + u9 * (slope_max - slope_min)
    jitter_v2 = jitter_min + u10 * (jitter_max - jitter_min)
    c_roll_sin3_v2 = c_roll_sin3_min + u11 * (c_roll_sin3_max - c_roll_sin3_min)
    c_roll_cos3_v2 = c_roll_cos3_min + u12 * (c_roll_cos3_max - c_roll_cos3_min)

    return [t_secondary, fp, offset_v1, slope_v1, jitter_v1, c_cy_v1, c_cx_v1, c_roll_sin2_v1,
            offset_v2, slope_v2, jitter_v2, c_roll_sin3_v2, c_roll_cos3_v2]


def loglike_joint(theta):

    jitters = [theta[4], theta[10]]

    _, model_flux_list, _, _ = model_joint_flux(theta, visits_norm)

    logL = 0.0

    for i, visit in enumerate(visits_norm):
        f = np.asarray(visit["flux"])
        fe = np.asarray(visit["flux_err"])

        resid = f - model_flux_list[i]
        sigma = fe * jitters[i]
        invvar = 1.0 / (sigma ** 2)

        logL += -0.5 * np.sum(resid * resid * invvar + np.log(2 * np.pi * sigma ** 2))

    return logL


# Run joint fit (dynesty version)
def run_dynesty_with_cache(model_name, param_names, loglike, prior, result_path):

    if os.path.exists(result_path):
        with open(result_path, "rb") as f:
            res = pickle.load(f)
        print(f"Loaded existing dynesty result from {result_path}")

    else:
        ndim = len(param_names)

        sampler = dynesty.NestedSampler(
            loglike,
            prior,
            ndim,
            nlive=400,
            bound="multi",
            sample="rwalk"
        )

        sampler.run_nested(dlogz=0.5)
        res = sampler.results

        try:
            with open(result_path, "xb") as f:
                pickle.dump(res, f)
                print(f"Saved dynesty result to {result_path}")
        except Exception as e:
            print(f"Could not save result to {result_path}: {e}")

    # corner plot
    try:
        samples = res.samples
        weights = np.exp(res.logwt - res.logz[-1])
        weights /= np.sum(weights)

        samples_equal = dyfunc.resample_equal(samples, weights)

        fig = corner.corner(samples_equal, labels=param_names, show_titles=True, title_fmt=".5f", quantiles=[0.16, 0.5, 0.84], 
                            title_kwargs={"fontsize": 14}, label_kwargs={"fontsize": 14}, tick_kwargs={"labelsize": 14}, max_n_ticks=4)

        # move the axis labels further from the tick labels
        for ax in fig.get_axes():
            # force label position farther from tick labels
            if ax.get_xlabel():
                ax.xaxis.set_label_coords(0.5, -0.5)
            if ax.get_ylabel():
                ax.yaxis.set_label_coords(-0.5, 0.5)
        
        # separate title names in 2 lines for better readability
        for ax in fig.get_axes():
            title = ax.get_title()
            if "=" in title:
                parts = title.split("=")
                new_title = "=\n".join(parts)
                ax.set_title(new_title, fontsize=14)

        plt.suptitle(f"Dynesty posterior samples for {model_name}")
        plt.show()

    except Exception as e:
        print("Failed to produce corner plot:", e)

    return res


param_names_joint = [
    "t_secondary", "fp",
    "offset_v1", "slope_v1", "jitter_v1", "c_cy_v1", "c_cx_v1", "c_roll_sin2_v1",
    "offset_v2", "slope_v2", "jitter_v2", "c_roll_sin3_v2", "c_roll_cos3_v2"
]

result_joint = run_dynesty_with_cache(
    "joint_all_visits",
    param_names_joint,
    loglike_joint,
    prior_joint,
    result_base
)

print("\n=== Joint Fit Summary ===")

try:
    samples = result_joint.samples
    weights = np.exp(result_joint.logwt - result_joint.logz[-1])
    weights /= np.sum(weights)

    theta_mean = np.average(samples, axis=0, weights=weights)
    theta_error = np.sqrt(np.average((samples - theta_mean) ** 2, axis=0, weights=weights))

    for n, val, err in zip(param_names_joint, theta_mean, theta_error):
        print(f"  {n}: {val:7f} ± {err:.7f}")

    logz = result_joint.logz[-1]
    logzerr = result_joint.logzerr[-1]
    print(f"\n  logZ: {logz:.7f} ± {logzerr:.7f}")

except Exception as e:
    print(f"Error extracting results: {e}")

In [ ]:
# Extract mean parameters from fitted results
def extract_theta_mean(result):
    """Extract posterior mean from dynesty result."""
    if result is None:
        return None

    try:
        samples = result.samples
        weights = np.exp(result.logwt - result.logz[-1])
        weights /= np.sum(weights)
        return np.average(samples, axis=0, weights=weights)
    except Exception:
        return None

# Extract theta_mean from result
theta_mean = extract_theta_mean(result_joint)

# Check if the fitted t_secondary is close to t0 + (n + 1/2) * P
if theta_mean is not None:
    # Use the same t0 and period as in the model
    t0 = t0_known
    P = P_known
    t_secondary_fit = theta_mean[0]

    print(f"Fitted t_secondary (shared): {t_secondary_fit:.8f}")

    if not visits:
        print("No visits loaded.")
    else:
        for v in visits:
            t_visit = np.asarray(v["time"], dtype=float)

            if t_visit.size == 0:
                print(f"\nVisit {v['name']}: no time points.")
                continue

            t_min = np.nanmin(t_visit)
            t_max = np.nanmax(t_visit)

            n_start = int(np.ceil(((t_min - t0) / P) - 0.5))
            n_end   = int(np.floor(((t_max - t0) / P) - 0.5))

            print(f"\nVisit {v['name']} time range: [{t_min:.8f}, {t_max:.8f}]")
            print("t_secondary values within this visit range:")

            candidates = []

            for n in range(n_start, n_end + 1):
                t_sec = t0 + (n + 0.5) * P
                candidates.append((n, t_sec))
                print(f"  n={n}: t_secondary={t_sec:.8f}")

            if candidates:
                n_nearest, t_secondary_expected = min(
                    candidates,
                    key=lambda x: abs(x[1] - t_secondary_fit)
                )
            else:
                # fallback: nearest epoch globally if none falls inside this visit window
                n_float = ((t_secondary_fit - t0) / P) - 0.5
                n_nearest = int(np.round(n_float))
                t_secondary_expected = t0 + (n_nearest + 0.5) * P
                print("  (No expected eclipse center falls inside this visit range.)")

            diff = t_secondary_fit - t_secondary_expected

            print(f"Nearest expected t_secondary: {t_secondary_expected:.8f}")
            print(f"Difference (fit - expected): {diff * 24 * 60:.4f} minutes")
            print(f"n used: {n_nearest}")

else:
    print("No fit result available.")

In [ ]:
# plot data for each visit with eclipse interval shaded (joint model)

theta_mean = extract_theta_mean(result_joint)

if theta_mean is not None:
      
    fp = theta_mean[1]  # eclipse depth parameter

    model_names, model_flux_list, bat_list, corr_list = model_joint_flux(theta_mean, visits_norm)

    for i, visit in enumerate(visits):
        t = np.asarray(visit["time"])
        f = np.asarray(visit["flux"])
        fe = np.asarray(visit["flux_err"])
        
        model_name = model_names[i]
        best_model = model_flux_list[i]
        bat = bat_list[i]

        # eclipse region from batman component (half depth criterion)
        thresh = np.max(bat) - 0.5 * fp
        eclipse_mask = bat < thresh
        if np.any(eclipse_mask):
            t_start, t_end = t[eclipse_mask].min(), t[eclipse_mask].max()
        else:
            t_start = t_end = None

        fig, ax = plt.subplots(figsize=(13, 5))
        ax.errorbar(t, f, yerr=fe, fmt=".", ms=3, alpha=0.6, label="data")
        ax.plot(t, best_model, color="C1", lw=1.5, label="joint model")
        ax.plot(t, bat, "--", color="C2", lw=1.0, label="batman eclipse component")

        if t_start is not None:
            ax.axvspan(t_start, t_end, color="gray", alpha=0.3, label="eclipse")
            ax.axvline(t_start, color="gray", ls="--", alpha=0.5)
            ax.axvline(t_end, color="gray", ls="--", alpha=0.5)

        ax.set_xlabel("BJD (days)")
        ax.set_ylabel("Normalized flux")
        ax.grid(alpha=0.3)
        ax.legend(loc="best", fontsize=11)
        plt.title(f"{PLANET} visit {i+1}\n{model_name}")
        plt.show()
else:
    print("No fit parameters available to mark eclipse.")


In [ ]:
# Allan deviation-style RMS vs binsize for residuals of each fitted model

if not visits:
    print("No visits loaded.")
else:
    # Helper: weighted posterior mean parameter vector
    def posterior_mean_theta(res):
        if res is None:
            return None

        try:
            samples = res.samples
            weights = np.exp(res.logwt - res.logz[-1])
            weights /= np.sum(weights)
            return np.average(samples, axis=0, weights=weights)
        except Exception:
            return None

    # Helper: RMS of residuals after binning by contiguous bins of size m
    def binned_rms(y, m):
        nbin = len(y) // m
        if nbin < 2:
            return np.nan
        y_trim = y[:nbin * m]
        y_bin = y_trim.reshape(nbin, m).mean(axis=1)
        return np.sqrt(np.mean((y_bin - np.mean(y_bin)) ** 2))

    # cadence in minutes (for secondary x-axis)
    cadence_min = np.nanmedian(np.diff(np.asarray(t))) * 24 * 60

    model_info = [
        ("joint_model", result_joint),
    ]

    fig, axes = plt.subplots(
        len(visits) // 2 + len(visits) % 2,
        2,
        figsize=(14, 6 * (len(visits) // 2 + len(visits) % 2)),
        squeeze=False
    )
    axes = axes.flatten()

    for i, ax in enumerate(axes):
        if i >= len(visits):
            ax.axis("off")

    for name, res in model_info:
        theta = posterior_mean_theta(res)

        if theta is None and "extract_theta_mean" in globals():
            theta = extract_theta_mean(res)

        if theta is None:
            for i, visit in enumerate(visits):
                axes[i].set_title(f"{name} visit {visit['name']}\n(no samples)")
                axes[i].axis("off")
            continue

        # Build model flux
        model_names, model_flux_list, bat_list, corr_list = model_joint_flux(theta, visits_norm)

        for i, visit in enumerate(visits):

            model_name = model_names[i]

            ax = axes[i]
            t = np.asarray(visit["time"], dtype=float)

            # `f` can be overwritten in other cells (e.g., `with open(...) as f`), so use visit flux explicitly
            flux_data = np.asarray(visit["flux"], dtype=float)
            model_flux = np.asarray(model_flux_list[i], dtype=float)

            resid = flux_data - model_flux
            good = np.isfinite(resid)
            resid = resid[good]

            # Use powers-of-two bins for a clean Allan/RMS curve
            max_m = max(2, len(resid) // 10)

            binsizes = []
            m = 1
            while m <= max_m:
                binsizes.append(m)
                m *= 2

            binsizes = np.array(binsizes, dtype=int)

            rms_vals = np.array([binned_rms(resid, m) for m in binsizes], dtype=float)
            valid = np.isfinite(rms_vals) & (rms_vals > 0)

            if np.any(valid):
                rms_vals_ppm = rms_vals[valid] * 1e6
                ax.loglog(binsizes[valid], rms_vals_ppm, "o-", ms=4, label="Residual RMS")

                # White-noise reference: RMS(1)/sqrt(m)
                rms1 = rms_vals[binsizes == 1][0] if np.any(binsizes == 1) else rms_vals[valid][0]
                ref = rms1 / np.sqrt(binsizes[valid])
                ax.loglog(binsizes[valid], ref * 1e6, "--", lw=1.2, label=r"$\propto N^{-1/2}$")

                # top x-axis: bin size converted to minutes
                axt = ax.twiny()
                axt.set_xscale("log")
                axt.set_xlim(ax.get_xlim())
                axt.set_xticks(binsizes[valid])
                axt.set_xticklabels([f"{(bs * cadence_min):.1f}" for bs in binsizes[valid]])
                axt.set_xlabel("Bin size (minutes)")

            ax.set_title(f"visit {visit['name']}\n{model_name}")
            ax.set_xlabel("Bin size (points)")
            ax.set_ylabel("RMS of binned residuals (ppm)")
            ax.set_yticks([60, 100, 200, 300])
            ax.set_yticklabels(["60", "100", "200", "300"])
            ax.grid(True, which="both", alpha=0.3)
            ax.legend()

    fig.suptitle(f"{PLANET} - joint model - Allan deviation / RMS-binning test")
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


In [ ]:
# Residuals vs systematics for each fitted model

if not visits:
    print("No visits loaded.")
else:
    systematics = [
        "time",
        "roll_angle",
        "centroid_x",
        "centroid_y",
        "background",
    ]

    def posterior_mean_theta(res):
        if res is None:
            return None

        try:
            samples = res.samples
            weights = np.exp(res.logwt - res.logz[-1])
            weights /= np.sum(weights)
            return np.average(samples, axis=0, weights=weights)
        except Exception:
            return None

    theta = posterior_mean_theta(result_joint)

    n_rows = len(visits)
    n_cols = len(systematics)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4 * n_cols, 2.8 * n_rows),
        squeeze=False
    )

    if theta is None:
        for i in range(n_rows):
            for j in range(n_cols):
                axes[i, j].text(0.5, 0.5, "No samples", ha="center", va="center")
                axes[i, j].set_axis_off()

    else:
        model_names, model_flux_list, bat_list, corr_list = model_joint_flux(theta, visits_norm)

        for i, visit in enumerate(visits):
            model_name = model_names[i]

            flux_data = np.asarray(visit["flux"], dtype=float)
            model_flux = np.asarray(model_flux_list[i], dtype=float)

            n = min(len(flux_data), len(model_flux))
            resid = flux_data[:n] - model_flux[:n]

            for j, sys_name in enumerate(systematics):
                ax = axes[i, j]

                sys_vals = np.asarray(
                    visit[sys_name] if sys_name != "time" else visit["time"],
                    dtype=float
                )

                x = sys_vals[:n]
                good = np.isfinite(x) & np.isfinite(resid)

                ax.plot(
                    x[good],
                    resid[good],
                    ".",
                    ms=3,
                    alpha=0.6,
                    color=COLORS[i % len(COLORS)]
                )

                ax.axhline(0.0, color="gray", ls="--", lw=1, alpha=0.7)
                ax.grid(True, alpha=0.25)

                if i == 0:
                    ax.set_title(sys_name)

                if j == 0:
                    ax.set_ylabel(f"visit {visit['name']}\nresidual")
                else:
                    ax.set_ylabel("")

                if i == n_rows - 1:
                    ax.set_xlabel(sys_name)

    fig.suptitle(f"{PLANET} — Residuals vs systematics (joint model)")

    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()